# 00 Source Log and Financial Extraction

Purpose: maintain an auditable source log and extraction checklist for an
Oracle public-equity valuation. This v1 model is source-aware and
offline-runnable: public facts seed the current setup, while the
assumptions remain easy to replace when filings are normalized.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    DcfInputs,
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    discounted_cash_flow,
    implied_margin_for_enterprise_value,
    implied_revenue_for_enterprise_value,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/oracle_valuation/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/oracle_valuation/data").exists()
)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
source_log = pd.read_csv(DATA_DIR / "source_log.csv")
source_log

## Current Fact Pattern

Oracle is being underwritten less like a mature software company and more
like a capital-intensive AI infrastructure buildout attached to a durable
database and applications franchise.

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
current_facts = (
    assumptions[
        assumptions["segment"].eq("Market")
        & assumptions["case"].eq("base")
        & assumptions["metric"].isin(
            [
                "current price",
                "current equity value",
                "FY2027 revenue guide",
                "FY2027 adjusted EPS guide",
                "remaining performance obligations",
                "FY2026 capex",
                "FY2026 free cash flow",
                "FY2027 gross capex guide",
                "FY2027 net capex cash outlay",
            ]
        )
    ]
    .loc[:, ["metric", "value", "unit", "source_id", "notes"]]
    .reset_index(drop=True)
)
current_facts

## Extraction Checklist

Replace seed assumptions as filings and company materials are normalized:

- consolidated revenue, operating income, capex, free cash flow and debt
- OCI revenue, cloud applications revenue, database support and license trends
- RPO roll-forward, customer concentration, prepayments and bring-your-own-hardware terms
- cash, debt, leases, equity issuance, diluted shares and SBC dilution
- data-center capacity, power availability, depreciation and mature OCI margin evidence
- consensus revenue, EPS, FCF and capex estimates through FY2030

In [ ]:
extraction_fields = pd.DataFrame(
    [
        ("Reported financials", "Revenue / operating margin / capex / FCF", "ORCL-Q4-FY2026"),
        ("Cloud infrastructure", "OCI revenue / capacity / RPO conversion / margin", "ORCL-Q4-FY2026"),
        ("Software durability", "Database support / applications / license migration", "ORCL-SEC"),
        ("Capital structure", "Cash / debt / leases / equity issuance / diluted shares", "ORCL-SEC"),
        ("Market data", "Price / market cap / EV bridge / forward EPS multiple", "MW-ORCL-PRICE"),
        ("Evidence gaps", "Consensus, customer concentration, mature OCI returns", "ANALYST-SEED"),
    ],
    columns=["workstream", "required_fields", "primary_source"],
)
extraction_fields